# 09.5 — Debugging a failing run

Sooner or later a sweep point ([09.4](09.4_parameter_sweeps.ipynb)) fails — the loss goes `NaN`, the job runs out of memory, training diverges, or a parity test breaks. This notebook is a **troubleshooting cookbook**: the four most common failure modes, how to *diagnose* each one, and the affordances the pipeline already provides to *fix* it. Most of these tie back to techniques from earlier modules — here they're assembled into a debugging workflow you can run down when a run goes wrong.

This notebook covers:

* NaN losses — where they come from and how to localize them.
* Out-of-memory — gradient accumulation as the escape hatch.
* Divergent training — gradient clipping and the usual suspects.
* Parity-test failures — reading a tolerance mismatch, the empirical-probe habit.

**Prerequisites:** [02.8 NaN handling](../02_numpy_and_pytorch_basics/02.8_nan_handling.ipynb), [05.2 gradient accumulation](../05_training_loop/05.2_gradient_accumulation.ipynb), [05.3 gradient clipping](../05_training_loop/05.3_gradient_clipping.ipynb).

## Section 1 — What MATLAB does

MATLAB's pipeline has relatively little built-in debugging tooling — you inspect variables in the debugger, add `fprintf` calls, and re-run. The Python port adds *structured* affordances for the common failures: NaN-safe loss kernels (Critical Note #38), hardware-aware gradient accumulation (Critical Note #18), gradient clipping wired to `GradientThreshold`, a pre-flight clobber check (Critical Note #22), and a tiered parity-test suite. This notebook is a tour of those, framed as "your run failed — now what?" Each fix is something the codebase already supports; the skill is *matching the symptom to the tool*.

## Section 2 — The Python concepts you need

### 2.1 — Symptom: the loss is `NaN`

A `NaN` loss is the most common — and most alarming — failure. It's contagious ([02.8](../02_numpy_and_pytorch_basics/02.8_nan_handling.ipynb)): once one appears, it spreads to every downstream value and the model's weights turn to `NaN`. The three usual sources:

* **`NaN` in the input** — a removed channel reached the encoder. Fix: `NaNToZero` before the network ([06.10 §2.1](../06_loss_orchestration/06.10_nan_masked_reconstruction.ipynb)).
* **`log(0)` or `0/0`** — a probability hit exactly 0 before a `log`, or an empty mask. Fix: the `eps` clamps in the loss kernels ([06.5](../06_loss_orchestration/06.5_mil_softmax_pooling.ipynb)).
* **Exploding gradients** — the loss blew up, the update overshot to `inf`, the next forward is `NaN`. Fix: gradient clipping (§2.3).

In [ ]:
import torch

# Diagnose: find WHERE the NaN enters. torch has tools to catch it.
x = torch.tensor([1.0, float("nan"), 3.0])
print("torch.isnan finds it:", torch.isnan(x).tolist())
print("any NaN in a tensor?", torch.isnan(x).any().item())

# A NaN input poisons a whole matmul (the classic 06.10 failure):
w = torch.ones(3, 3)
print("matmul with a NaN input → all NaN?", torch.isnan(x @ w).all().item())

# The fix for input NaNs — replace them BEFORE the network:
from neural_data_decoding.models.layers.nan_to_zero import NaNToZero
print("after NaNToZero, any NaN?", torch.isnan(NaNToZero()(x)).any().item())

**Diagnosis workflow:** add `assert not torch.isnan(t).any()` at suspect points (input, after the encoder, after each loss term) to *bisect* where the `NaN` first appears — the earliest failing assert localizes the source. Then apply the matching fix. `torch.autograd.set_detect_anomaly(True)` will even point at the operation that produced a `NaN` in the *backward* pass (slow, so use it only while debugging).

### 2.2 — Symptom: out of memory (OOM)

In [ ]:
from neural_data_decoding.training.accumulation import micro_batch_chunks

# OOM = the batch doesn't fit in GPU memory. The fix: process it in MICRO-BATCHES
# and accumulate gradients, so the effective batch is large but memory stays small.
big_batch = 100
max_that_fits = 30
chunks = list(micro_batch_chunks(big_batch, max_that_fits))
print(f"a batch of {big_batch} that won't fit → process in chunks of ≤{max_that_fits}:")
for start, end, weight in chunks:
    print(f"   samples [{start:3}:{end:3}]  weight {weight}")
print(f"weights sum to {sum(w for _, _, w in chunks):.1f} → the accumulated gradient equals the full-batch gradient.")

Out-of-memory means the batch is too big for the GPU. The fix isn't a smaller *effective* batch (that changes the training dynamics) — it's **gradient accumulation** ([05.2](../05_training_loop/05.2_gradient_accumulation.ipynb)): split the batch into micro-batches that *do* fit, run each forward/backward, weight each by its fraction, and accumulate the gradients before one optimizer step. The result is *mathematically identical* to the full batch, at a fraction of the memory. The weights (`0.3, 0.3, 0.3, 0.1` above) sum to 1 so the accumulated gradient matches the full-batch gradient exactly. The pipeline even *auto-sizes* the micro-batch to the detected hardware:

In [ ]:
from neural_data_decoding.training.accumulation import get_accumulation_size_for_current_system

# A hardware→max-micro-batch table (Critical Note #18). The pipeline looks up the
# entry for the detected GPU; on a host with no matching GPU it returns None.
hw_table = {"A100": 100, "V100": 50, "GTX 1080": 25}
chosen = get_accumulation_size_for_current_system(hw_table)
if chosen is None:
    print("detected-hardware micro-batch cap: None (no matching GPU here → no cap, use full batch)")
else:
    print(f"detected-hardware micro-batch cap: {chosen}")
print("→ the same config adapts: on an A100 it caps at 100, on a smaller GPU less, on CPU no cap.")

### 2.3 — Symptom: training diverges (loss explodes or oscillates)

In [ ]:
# Divergence often = gradients exploding. Gradient clipping caps the gradient NORM,
# rescaling (not truncating) so the DIRECTION is preserved (05.3).
torch.manual_seed(0)
model = torch.nn.Linear(4, 4)
for p in model.parameters():
    p.grad = torch.randn_like(p) * 1000.0        # simulate a huge (exploding) gradient

pre = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)   # clips IN PLACE, returns pre-clip norm
post = torch.sqrt(sum((p.grad ** 2).sum() for p in model.parameters()))  # measure the ACTUAL post-clip norm
print(f"gradient norm BEFORE clipping: {pre:.0f}  (exploding)")
print(f"gradient norm AFTER  clipping: {post:.2f}  (capped at max_norm=5.0)")
print("→ the norm is rescaled down to 5.0 — same direction, safe magnitude, no overshoot.")

Divergence — the loss shoots to `inf` or oscillates wildly — usually means the updates are too big. The checklist:

* **Exploding gradients** → gradient clipping ([05.3](../05_training_loop/05.3_gradient_clipping.ipynb)), wired to `cfg.gradient_threshold`. It caps the gradient *norm*, rescaling so the direction survives.
* **Learning rate too high** → lower `initial_learning_rate`, or check the LR schedule ([05.4](../05_training_loop/05.4_learning_rate_scheduling.ipynb)).
* **Posterior collapse / KL too strong too early** → the KL annealing warmup ([07.4 §2.4](../07_dynamic_curriculum/07.4_loss_weights_curriculum.ipynb)).
* **One loss component dominating** → the EMA normalization ([06.4](../06_loss_orchestration/06.4_the_ema_prior_normalization_deep_dive.ipynb)); if it's off, the largest-scale loss can bulldoze the rest.

Change *one* thing at a time (like the sweep itself, [09.4 §2.4](09.4_parameter_sweeps.ipynb)) so you know what fixed it.

### 2.4 — Symptom: a parity test fails

In [ ]:
import numpy as np

# Parity tests compare Python output to a MATLAB golden value with a tolerance.
# A failure prints the max abs difference — read it to judge real bug vs float noise.
python_value = np.array([1.0000001, 2.0, 3.0])
matlab_golden = np.array([1.0, 2.0, 3.0])
try:
    np.testing.assert_allclose(python_value, matlab_golden, rtol=1e-6, atol=1e-6)
    print("parity holds within tolerance.")
except AssertionError as e:
    line = [l for l in str(e).splitlines() if "Max absolute" in l or "Mismatch" in l]
    print("parity FAILED:", line[0].strip() if line else "(see full diff)")
    print("→ max diff ~1e-7 here — within float noise, so LOOSEN atol or accept; a 1e-2 diff = real bug.")

A parity-test failure ([06.10](../06_loss_orchestration/06.10_nan_masked_reconstruction.ipynb)) means the Python output drifted from the MATLAB golden value beyond the tolerance (`rtol=atol=1e-6` for the numeric tiers). The failure message reports the *max absolute difference* — read it:

* **~1e-7 to 1e-6** — floating-point noise (different summation order, BLAS libraries). Usually not a real bug; the tolerance may be slightly tight.
* **~1e-3 or larger** — a *real* divergence. Something in the port doesn't match MATLAB. This is where the **empirical-probe discipline** ([06.10 §2.4](../06_loss_orchestration/06.10_nan_masked_reconstruction.ipynb)) pays off: don't guess — probe the MATLAB function directly to find the true behavior. That's how the reconstruction-normalization bug was caught. And remember the `needs_matlab` tier is skipped by default; run it explicitly to check actual MATLAB consumption ([08.4](../08_output_and_analysis/08.4_the_mat_round_trip_test.ipynb)).

### 2.5 — Before the run: the pre-flight clobber check

In [ ]:
from neural_data_decoding.training.checkpoint import has_existing_checkpoint
from pathlib import Path
import tempfile

# The BEST debugging is not clobbering good results. `check-existing` / the --force
# guard detect a directory that already has a completed run (Critical Note #22).
empty = Path(tempfile.mkdtemp())
print("empty dir has an existing checkpoint?", has_existing_checkpoint(empty))
(empty / "optimal_state.pt").write_bytes(b"")     # simulate a finished run
print("after a run wrote optimal_state.pt?     ", has_existing_checkpoint(empty))
print("→ `train` aborts on a non-empty dir unless --force; `check-existing` reports it for sweeps.")

The cheapest bug to fix is the one you prevent. Before training, `train` checks whether the target directory already holds a completed run (`has_existing_checkpoint`, Critical Note #22) and *aborts unless you pass `--force`* — so a re-submitted sweep can't silently overwrite hours of results. The `check-existing` subcommand does the same check as a pre-flight (returning exit code 1 if a run exists), so a sweep script can *skip* already-done points instead of redoing them. Combined with the resume logic ([05.5](../05_training_loop/05.5_checkpoint_resume_state_machine.ipynb)), this makes a big sweep restartable and idempotent.

## Section 3 — The `neural_data_decoding` implementation

`micro_batch_chunks` — the OOM escape hatch, weights that preserve the gradient:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/training/accumulation.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if line.startswith("def micro_batch_chunks"))
for k in range(i, i + 6):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* `micro_batch_chunks(n_total, max_size)` yields `(start, end, weight)` triples — each a slice of the batch and its fraction of the whole. `max_size=None` means "it all fits" (one chunk, weight 1).
* The weights sum to 1 so the accumulated gradient equals the full-batch gradient (§2.2) — the mathematical guarantee that makes accumulation *transparent*, not an approximation.
* `get_accumulation_size_for_current_system` reads a `{hardware: max_size}` table and picks the entry for the detected GPU (or the min as a fallback) — Critical Note #18, ported from `cgg_getAccumulationSizeForCurrentSystem.m`.
* `has_existing_checkpoint` (in `training/checkpoint.py`) is the pre-flight guard (§2.5, Critical Note #22); the NaN-safe loss kernels are in `training/losses/elbo.py` (§2.1, Critical Note #38).
* None of these are debugging *afterthoughts* — they're first-class parts of the pipeline, because a production sweep of thousands of runs *will* hit every one of these failures, and the tooling has to handle them without a human watching each job.

## Section 4 — Hands-on exercises

### Exercise 4.1 — bisect a NaN

Given a mini "forward pass" where a NaN appears partway through, use `torch.isnan` asserts to find the first stage that produces it.

In [ ]:
# Reveal:
def stage_a(x): return x * 2
def stage_b(x): return x / torch.tensor([1.0, 0.0, 1.0])   # divide by zero → inf/nan here
def stage_c(x): return x + 1

x = torch.tensor([1.0, 2.0, 3.0])
for name, fn in [("stage_a", stage_a), ("stage_b", stage_b), ("stage_c", stage_c)]:
    x = fn(x)
    bad = torch.isnan(x).any() or torch.isinf(x).any()
    print(f"after {name}: has NaN/inf? {bool(bad)}")
    if bad:
        print(f"→ the NaN/inf FIRST appears in {name} — that's where to look (the /0).")
        break

### Exercise 4.2 — accumulation is exact, not approximate

Show that summing the weighted micro-batch means equals the full-batch mean — so accumulation changes memory, not math.

In [ ]:
# Reveal:
data = torch.randn(100, 4)
full_mean = data.mean(dim=0)
acc_mean = sum(data[s:e].mean(dim=0) * w for s, e, w in micro_batch_chunks(100, 30))
print("max |full-batch mean − accumulated mean|:", (full_mean - acc_mean).abs().max().item())
print("→ ~0: gradient accumulation is mathematically identical to the full batch.")

## Section 5 — Systematic debugging (meta)

Beyond the specific fixes, a workflow that works for *any* failing run:

* **Reproduce it minimally.** Shrink to the smallest config that still fails (fewer epochs, tiny data). A fast repro is the difference between 10 iterations an hour and 1.
* **Read the actual error, fully.** The traceback names the failing op and file:line. The `NaN`/OOM/parity messages carry the diagnostic number (which stage, how much memory, how big the diff) — don't skim past it.
* **Bisect.** Whether it's a `NaN` (which stage, §2.1) or a regression (which commit — `git bisect`, [00.6](../00_orientation/00.6_git_and_github_for_matlab_users.ipynb)), halve the search space each step.
* **Change one thing at a time.** Like the sweep ([09.4](09.4_parameter_sweeps.ipynb)) — if you change three things and it works, you don't know which fixed it.
* **Verify empirically, don't guess.** The reconstruction-normalization bug ([06.10](../06_loss_orchestration/06.10_nan_masked_reconstruction.ipynb)) was invisible to reasoning and obvious to a probe. When porting, the MATLAB fixture is the source of truth, not your mental model.
* **Check the cheap things first.** Wrong environment/paths ([09.1](09.1_environment_detection.ipynb)), a clobbered directory (§2.5), a typo'd override key ([09.3](09.3_hydra_config_composition.ipynb)) — these masquerade as deep bugs.

## Section 6 — Further reading

- [`src/neural_data_decoding/training/accumulation.py`](../../src/neural_data_decoding/training/accumulation.py) — `micro_batch_chunks`, `get_accumulation_size_for_current_system`.
- [`src/neural_data_decoding/training/checkpoint.py`](../../src/neural_data_decoding/training/checkpoint.py) — `has_existing_checkpoint` (the clobber guard).
- [`torch.autograd.set_detect_anomaly`](https://pytorch.org/docs/stable/autograd.html#torch.autograd.set_detect_anomaly) — pinpoint a NaN in the backward pass.

Next up: **[09.6 extending the pipeline](09.6_extending_the_pipeline.ipynb)** — the final notebook: how to add a new architecture, loss, curriculum, or target task.